**[🏠 Course Home](../README.md) | 🐍 Python Track: [Sheet: Two-Sample Comparison](../python/sheet_t_test_frequentist_vs_bayesian.ipynb) | 🔵 R Track: [Sheet: Two-Sample Comparison](../r/sheet_t_test_frequentist_vs_bayesian.ipynb) | ↩️ Previous: [Sheet: Bayesian Mean](sheet_calculating_a_mean_in_a_bayesian_world.ipynb)**

---

# 📊 Applied Conceptual Guide: Comparing Two Groups — The $t$-Test
### *From Null Hypothesis Significance Testing (NHST) to Bayes Factors & Bayesian Estimation (BEST)*

---

## 🧭 Executive Summary: The A/B Testing Dilemma

Every software engineer and product team runs two-sample comparisons:
* *Variant A (Baseline Control)* vs. *Variant B (New Pipeline / Redesigned UI)*.
* Did Variant B reduce latency? Did it improve conversion? Or was the observed gain just a fluke of random noise?

For 90 years, the default tool for this job has been **Null Hypothesis Significance Testing (NHST)** and Student's/Welch's $t$-test. But in production engineering, classical NHST suffers from three catastrophic conceptual failure modes.

---

## 1. The Three Fatal Fallacies of NHST & The $p$-Value

### Fallacy 1: The Large-$N$ Trap (Statistical vs. Practical Significance)
In modern web applications, experiments easily collect $N = 500{,}000$ requests.
* Because the $t$-statistic is proportional to $\sqrt{N}$, **any difference—no matter how microscopic—will eventually yield $p < 0.001$**.
* If Variant B is faster by $0.0002\text{ ms}$ (a difference completely imperceptible to users and dwarfed by network jitter), a frequentist $t$-test will declare:
  > *"Statistically significant improvement! Reject the null hypothesis!"*
* Teams end up celebrating and deploying changes that have zero real-world value.

### Fallacy 2: The Illusion of Posterior Probability
Engineers almost universally misinterpret $p = 0.03$ as meaning:
> ❌ *"There is only a 3% chance that the null hypothesis is true (or a 97% chance Variant B is better)."*
* **The Reality:** A $p$-value is $P(\text{Data as extreme as observed} \mid H_0)$. It is a statement about hypothetical data under an assumed null universe. It tells you **nothing** about $P(H_0 \mid \text{Data})$.

### Fallacy 3: Absence of Evidence $\neq$ Evidence of Absence
When $p = 0.18$, what can the frequentist say?
* Only that we *"fail to reject the null hypothesis."*
* We cannot say Variant A and Variant B are equal! The test might simply be underpowered. Frequentist NHST cannot accept the null hypothesis.

---

## 2. The Bayesian Revolution: Two Complementary Routes

To escape these traps, Bayesian statistics offers two clean, principled routes:

```
                               Two-Sample Comparison
                                        |
                 +----------------------+----------------------+
                 |                                             |
       Route 1: Model Comparison                     Route 2: Parameter Estimation
         (Bayes Factor BF_10)                             (Kruschke's BEST)
                 |                                             |
   "Which hypothesis is better supported:          "What is the actual magnitude of the
    the difference is zero (M_0) or non-zero (M_1)?" difference, its uncertainty, and its ROPE?"
```

---

## 3. Route 1: The Courtroom Scale (JZS Bayes Factor)

When executives demand a binary decision (*"Should we ship this feature or kill it?"*), model comparison uses the **Bayes Factor ($BF_{10}$)**:

$$BF_{10} = \frac{P(y \mid \mathcal{M}_1)}{P(y \mid \mathcal{M}_0)}$$

* $BF_{10} = 1$: Data is equally likely under both models (neutral).
* $BF_{10} = 10$: The data are $10\times$ more likely under the alternative model $\mathcal{M}_1$ than under the null $\mathcal{M}_0$.
* $BF_{10} = 0.1$: The data are $10\times$ more likely under the **null** model ($BF_{01} = 10$). **Bayes factors can provide direct positive evidence for the null!**

### The Jeffreys-Zellner-Siow (JZS) Prior
How do we define the alternative model without artificially gaming the test?
* The **JZS prior** places a Cauchy distribution with scale $r = \frac{\sqrt{2}}{2} \approx 0.707$ on the standardized effect size $\delta$.
* It is scale-invariant, symmetric, and acts as an impartial benchmark endorsed across scientific disciplines.

---

## 4. Route 2: Kruschke's BEST & The ROPE Decision Framework

In engineering, the null hypothesis is usually pointless:
> *"Is the latency difference between Version A and Version B exactly 0.00000000 ms?"*  
> Almost certainly not. Every code commit touches instruction caches and memory layouts.

Instead of testing whether a difference is zero, **Bayesian Estimation Supersedes the $t$-Test (BEST)** models the two groups directly:
1. It estimates $\mu_A, \mu_B$ and independent standard deviations $\sigma_A, \sigma_B$.
2. It yields the complete posterior distribution of difference $\Delta\mu = \mu_B - \mu_A$.

### The ROPE Ruler (Region of Practical Equivalence)
A **ROPE** defines the zone of operational indifference (e.g. any latency difference within $[-2.0\text{ ms}, +2.0\text{ ms}]$ is too small to matter).

We make decisions by overlaying the **95% Highest Density Interval (HDI)** onto the **ROPE**:

```
 Case 1: Meaningful Improvement (SHIP IT)
               [--- 95% HDI ---]
 --------------------------------|=== ROPE ===|------------------> Δμ
                               -2             +2

 Case 2: Proven Equivalence (NO MEANINGFUL DIFFERENCE)
                                  [ 95% HDI ]
 --------------------------------|=== ROPE ===|------------------> Δμ
                               -2             +2

 Case 3: Inconclusive (COLLECT MORE DATA)
                         [------ 95% HDI ------]
 --------------------------------|=== ROPE ===|------------------> Δμ
                               -2             +2
```

1. **Meaningful Win:** 95% HDI lies entirely outside the ROPE to the left ($P(\Delta\mu < -2\text{ ms}) > 95\%$).
2. **True Practical Equivalence:** 95% HDI lies entirely inside the ROPE ($P(|\Delta\mu| \le 2\text{ ms}) > 95\%$).
3. **Inconclusive:** 95% HDI overlaps the ROPE boundary. Continue testing.

---

## 5. The Peeking / Optional Stopping Trap

In fast-paced tech companies, teams check dashboards continuously:
* If $p < 0.05$ on Tuesday, they stop the test and declare victory.
* If not, they keep running until Friday.

### The Disaster of Frequentist Peeking:
* In frequentist statistics, stopping rules are baked into the mathematics of the $p$-value.
* Repeatedly peeking and stopping at the first sign of $p < 0.05$ inflates the false positive rate from $5\%$ to **$30\% - 60\%$**!

### Why Bayesian Updating is Immune:
* In Bayesian inference, evidence is evaluated condition-by-condition:
  $$p(\theta \mid y_1, y_2) \propto p(y_2 \mid \theta) \, p(\theta \mid y_1)$$
* You can inspect the posterior after every single sample. If you stop because the 95% HDI has shrunken entirely inside the ROPE, your conclusion remains mathematically valid because your posterior represents your updated state of belief given all data observed so far.

---

## 🏁 Summary Checklist: Which Route Should You Choose?

| Scenario | Recommended Approach | Reason |
| :--- | :--- | :--- |
| **Strict regulatory / journal publication** | Welch's $t$-test (Frequentist) | Fixed pre-registered sample size; standardized peer-review expectation. |
| **Direct evidence for equality ($A = B$)** | JZS Bayes Factor ($BF_{01}$) | Quantifies exact evidence supporting the null over the alternative. |
| **Production A/B testing & decision making** | Kruschke's BEST + ROPE | Prevents the large-$N$ trap; models effect sizes; enables continuous monitoring. |

---

* 🐍 **Explore the Runnable Python Lab**: [Sheet: Two-Sample Comparison (t-Test)](../python/sheet_t_test_frequentist_vs_bayesian.ipynb)
* 🔵 **Explore the Runnable R Lab**: [Sheet: Two-Sample Comparison (t-Test)](../r/sheet_t_test_frequentist_vs_bayesian.ipynb)

